# To AI or Not To AI: training notebook

This notebook trains the detector end to end:

1. Builds a clean, label-checked dataset from the attached Kaggle datasets, checks the file formats per class,
   and removes test images that duplicate a training image.
2. Trains with the repository's own `scripts/train.py`, so the code that trained the model is the committed code.
3. Checks the ONNX export against the PyTorch model.
4. Evaluates once on a held-out test set, overall, per category, per source, and on a generator never seen in training.
5. Chooses the uncertainty thresholds from the validation set and reports how often the model decides.
6. Checks robustness to JPEG, resizing and blur, and draws a GradCAM grid to see what the model looks at.
7. Packages the model files, reports and checksums for download.

**On Kaggle:** attach the datasets listed in the configuration cell, choose a GPU (T4) accelerator, turn Internet on, then *Run All*. Expect about 2 hours.

**Rules this notebook follows:** one seed for everything; labels come from explicit rules that are printed for inspection; no single source can dominate; tiny images are dropped; the test set is never used for training or model selection.

> **Note:** this notebook clones the commit in `REPO_REF`, so its imports use that commit's file names.
> On newer commits the engine lives in `src.ml.predictor` and `src.ml.familiarity`,
> and the training dependencies are in `requirements/train.txt`. Update the imports if you change `REPO_REF`.

In [ ]:
# Setup: clone the repository at a fixed ref and install what Kaggle lacks.
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/gismal/to-ai-or-not-to-ai.git"
REPO_REF = "915e10c6bc01a5e2a2d1609906fb645775c50d76"   # a branch, tag or commit; the exact commit is recorded below

%cd /kaggle/working
!rm -rf repo && git clone --quiet {REPO_URL} repo && cd repo && git checkout --quiet {REPO_REF}
%cd /kaggle/working/repo
!pip install -q onnx onnxruntime onnxscript pydantic-settings python-dotenv imagehash grad-cam

GIT_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("Training code at commit", GIT_COMMIT)

# src.config needs these values to import. They are placeholders, not secrets.
Path(".env").write_text(
    "API_KEY=training-only\n"
    "DATABASE_URL=sqlite+aiosqlite:///unused.db\n"
    "MODEL_THRESHOLD=0.75\n"
    "GRAY_AREA_MARGIN=0.35\n"
)

In [ ]:
# Configuration: everything you might want to change lives in this cell.
import os
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

SEED = 42
FACE_CAP = 12_000     # images per class from face datasets
GENERAL_CAP = 30_000    # images per class from everything else
SPLIT = (0.8, 0.1, 0.1) # train / val / test
MIN_SIDE = 128          # smaller images are dropped: they teach "low resolution means real"

# Sources used only for testing, never for training: how does the model do on a dataset it has
# never seen? Use names from SOURCES below. A recent-generator dataset would be the strictest test.
HOLDOUT_SOURCES: tuple[str, ...] = ("RealFakeFace", "FaceForensics")
HOLDOUT_CAP = 2_000         # test images per class from the held-out sources
TARGET_PRECISION = 0.95     # a definite verdict should be right this often; sets the thresholds
ROBUSTNESS_SAMPLE = 2_000   # test images used for the robustness check
INPUT_ROOT = Path(os.getenv("AON_INPUT_ROOT", "/kaggle/input"))
DATA_DIR = Path("data")
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp"}

random.seed(SEED)
rng = random.Random(SEED)


@dataclass(frozen=True)
class Source:
    name: str
    keywords: tuple[str, ...]            # matched against the mounted dataset folder name
    category: str                        # "face" or "general"
    force_label: str | None = None       # set for datasets that contain only one class
    include_dirs: tuple[str, ...] = ()   # if set, only images under these exact folder names
    csv: dict | None = None              # labels from a CSV instead of folder names


SOURCES = [   # names in HOLDOUT_SOURCES must match a name here
    Source("LFW", ("labelled-faces", "labeled-faces", "lfw"), "face", force_label="REAL"),
    Source("CelebA", ("celeba", "celebfaces"), "face", force_label="REAL"),
    Source("FFHQ", ("flickr-faces", "ffhq"), "face", force_label="REAL"),
    Source("Selfie2Anime", ("selfie2anime",), "face", force_label="REAL",
           include_dirs=("trainA", "testA")),   # the selfie half only
    Source("Deepfake", ("deepfake-image-detection",), "face"),
    Source("DFDC", ("dfdc",), "face"),
    Source("FaceForensics", ("faceforensic",), "face"),
    Source("RealFakeFace", ("real-and-fake-face",), "face"),
    Source("AIvsHuman", ("ai-vs-human-generated",), "general",
           csv={"file": "train.csv", "file_col": "file_name", "label_col": "label",
                "ai_values": {"1", "1.0"}}),
    Source("AIvsReal", ("ai-generated-images-vs-real",), "general"),
    Source("OpenImagesV7", ("open-images-v7",), "general", force_label="REAL"),
    # Add a dataset of images from recent generators here, for example:
    # Source("RecentGenerators", ("<dataset-folder-keyword>",), "general"),
]
# Left out on purpose: CIFAKE (32x32 images) and the AI-vs-human *art* dataset (a different task).

## 1. Find and label every image

For datasets that contain both classes, the label comes from the **nearest folder whose name says real or AI**, for example `training_real`, `fake` or `manipulated_sequences`. Names are split into words, so `train` never counts as "ai". A folder that mentions both classes, such as `real_vs_fake`, is treated as a container and skipped. Images whose path never names a class are left out and counted as *unlabeled*.

**Read the output of the next cell before training.** It lists which folder names decided each label. If a name looks wrong, add a word to `REAL_WORDS` or `AI_WORDS`, or set `include_dirs`, and run it again.

In [ ]:
import pandas as pd

REAL_WORDS = {"real", "human", "authentic", "original", "genuine", "live", "pristine"}
AI_WORDS = {"fake", "ai", "generated", "synthetic", "manipulated", "spoof", "deepfake", "deepfakes"}


def find_root(keywords):
    """The shallowest mounted folder whose name contains a keyword.

    Kaggle mounts datasets as /kaggle/input/<dataset> or /kaggle/input/datasets/<user>/<dataset>,
    so up to three levels are searched, shallowest first.
    """
    for d in [*sorted(INPUT_ROOT.glob("*")), *sorted(INPUT_ROOT.glob("*/*")), *sorted(INPUT_ROOT.glob("*/*/*"))]:
        if d.is_dir() and any(k in d.name.lower() for k in keywords):
            return d
    return None


def iter_images(root, include_dirs=()):
    """Every image under root in a stable order, optionally limited to some folder names."""
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        parts = Path(dirpath).relative_to(root).parts
        if include_dirs and not any(p in include_dirs for p in parts):
            continue
        for f in sorted(filenames):
            if Path(f).suffix.lower() in IMAGE_EXT:
                yield Path(dirpath) / f


def words(name):
    return set(re.split(r"[^a-z0-9]+", name.lower())) - {""}


def label_from_path(path, root):
    """The nearest folder naming exactly one class decides. Returns (label, folder) or (None, None)."""
    for part in reversed(path.relative_to(root).parent.parts):
        w = words(part)
        is_real, is_ai = bool(w & REAL_WORDS), bool(w & AI_WORDS)
        if is_real != is_ai:
            return ("REAL" if is_real else "AI_GENERATED"), part
    return None, None


def labels_from_csv(src, root):
    """{path: label} from a labels CSV, resolving file names relative to the CSV's folder."""
    cfg, out = src.csv, {}
    for csv_path in sorted(root.rglob(cfg["file"])):
        df = pd.read_csv(csv_path, dtype=str)
        base = csv_path.parent
        for fname, value in zip(df[cfg["file_col"]], df[cfg["label_col"]]):
            for cand in (base / fname, base / "images" / fname, base / "train" / fname, base / "train_data" / fname):
                if cand.exists():
                    out[cand] = "AI_GENERATED" if str(value).strip() in cfg["ai_values"] else "REAL"
                    break
    return out


pools = defaultdict(list)   # (category, label) -> [(source, path), ...]
inventory, deciders = [], {}
for src in SOURCES:
    root = find_root(src.keywords)
    if root is None:
        inventory.append({"source": src.name, "category": src.category, "folder": "NOT FOUND"})
        continue
    counts, decided_by = Counter(), Counter()
    if src.csv:
        found = labels_from_csv(src, root).items()
        decided_by.update((label, f"CSV {src.csv['file']}") for _, label in found)
    else:
        found = []
        for path in iter_images(root, src.include_dirs):
            label, why = (src.force_label, "(whole dataset)") if src.force_label else label_from_path(path, root)
            if label is None:
                counts["unlabeled"] += 1
                continue
            found.append((path, label))
            decided_by[(label, why)] += 1
    for path, label in found:
        pools[(src.category, label)].append((src.name, path))
        counts[label] += 1
    inventory.append({"source": src.name, "category": src.category,
                      "folder": str(root.relative_to(INPUT_ROOT)), "REAL": counts["REAL"],
                      "AI_GENERATED": counts["AI_GENERATED"], "unlabeled": counts["unlabeled"]})
    deciders[src.name] = decided_by.most_common(6)

print(pd.DataFrame(inventory).fillna(0).to_string(index=False))
print("\nWhich folder names decided the labels (check these!):")
for name, top in deciders.items():
    print(f"  {name:14s} " + ", ".join(f"{label} <- '{why}' ({n:,})" for (label, why), n in top))

for category in ("face", "general"):
    for label in ("REAL", "AI_GENERATED"):
        assert pools[(category, label)], f"No {label} images for '{category}': fix the sources above"

### File formats per class

If one class is mostly PNG and the other mostly JPEG, the model can learn "is this JPEG-compressed?" instead of "is this AI?". It then scores well here and fails on the web, where AI images are re-saved as JPEG. `train.py` re-compresses training images as JPEG to break that shortcut; this table shows how big the risk is in this data.

In [ ]:
from PIL import Image

check = random.Random(SEED)
fmt_rows = []
for (category, label), items in sorted(pools.items()):
    formats = Counter()
    for _, path in check.sample(items, min(300, len(items))):
        try:
            with Image.open(path) as im:
                formats[im.format or "unknown"] += 1
        except Exception:
            formats["unreadable"] += 1
    total = sum(formats.values())
    fmt_rows.append({"category": category, "label": label, "sampled": total,
                     **{f: round(n / total, 2) for f, n in formats.most_common()}})
format_check = pd.DataFrame(fmt_rows).fillna(0)
print(format_check.to_string(index=False))

Path("reports").mkdir(exist_ok=True)
format_check.to_csv("reports/format_check.csv", index=False)
for category, g in format_check.groupby("category"):
    jpeg = dict(zip(g.label, g["JPEG"] if "JPEG" in g else [0] * len(g)))
    gap = abs(jpeg.get("REAL", 0) - jpeg.get("AI_GENERATED", 0))
    verdict = "WARNING: keep JPEG re-compression on in train.py" if gap > 0.2 else "similar"
    print(f"{category}: JPEG share differs by {gap:.0%} between the classes -> {verdict}")

In [ ]:
# Optional: look at two random images per source and label before trusting the table above.
import matplotlib.pyplot as plt
from PIL import Image

groups = defaultdict(list)
for (_, label), items in pools.items():
    for source, path in items:
        groups[(source, label)].append(path)
view = random.Random(SEED)
keys = sorted(groups)
fig, axes = plt.subplots(len(keys), 2, figsize=(5, 2.3 * len(keys)), squeeze=False)
for row, key in zip(axes, keys):
    for ax, path in zip(row, view.sample(groups[key], min(2, len(groups[key])))):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(f"{key[0]} / {key[1]}", fontsize=8)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Sample, balance and split

* Sources take turns, so no single dataset dominates a class.
* Images under `MIN_SIDE` pixels and unreadable files are skipped while sampling.
* Within each category, both classes get the same number of images. Otherwise "face" would become a hint for one label.
* Sources in `HOLDOUT_SOURCES` never reach train or val; up to `HOLDOUT_CAP` of their images per class go to test only.
* **Duplicates:** every image gets the API's own perceptual hash (`src.utils.generate_phash`). A val or test image whose hash already appears in an earlier split is removed, so the model is never graded on a copy of an image it trained on.
* The split is 80 / 10 / 10. `data/manifest.csv` records every image's split, label, category and source, which is what the per-source evaluation uses.

In [ ]:
import csv
import shutil
import warnings
from concurrent.futures import ThreadPoolExecutor

import imagehash
from PIL import Image
from tqdm.auto import tqdm

MAX_PIXELS = 50_000_000   # bigger files are rare, slow to decode, and get shrunk to 224 px for training anyway
warnings.simplefilter("ignore", Image.DecompressionBombWarning)   # we filter the giants ourselves below


def usable(path):
    """Big enough to learn from, small enough to decode quickly. Image.open only reads the header."""
    try:
        with Image.open(path) as im:
            w, h = im.size
            return min(w, h) >= MIN_SIDE and w * h <= MAX_PIXELS
    except Exception:
        return False   # unreadable files are dropped as well

GIVE_UP_AFTER = 300   # this many unusable files in a row means the source is mostly too small: stop trying it

CHECKER = ThreadPoolExecutor(16)


def checked(paths, batch=64):
    """Yields (path, usable?) in order, checking a batch of files at a time in parallel."""
    for i in range(0, len(paths), batch):
        chunk = paths[i:i + batch]
        yield from zip(chunk, CHECKER.map(usable, chunk))


def round_robin(items, cap, skipped):
    """Up to `cap` usable images, taken in turn from every source."""
    by_source = defaultdict(list)
    for source, path in items:
        by_source[source].append(path)
    streams = {}
    for source, paths in by_source.items():
        rng.shuffle(paths)
        streams[source] = checked(paths)
    picked, misses, active = [], Counter(), sorted(streams)
    while len(picked) < cap and active:
        for source in list(active):
            for path, ok in streams[source]:
                if ok:
                    picked.append((source, path))
                    misses[source] = 0
                    break
                skipped[source] += 1
                misses[source] += 1
                if misses[source] >= GIVE_UP_AFTER:
                    print(f"  {source}: {GIVE_UP_AFTER} unusable files in a row, skipping the rest of it")
                    streams[source] = iter(())
                    break
            else:                       # this source ran out of files
                active.remove(source)
            if len(picked) >= cap:
                break
    return picked

def phash(path):
    """Perceptual hash of a small copy. JPEGs decode straight at reduced size, so this is fast and light."""
    try:
        with Image.open(path) as im:
            im.draft("RGB", (512, 512))
            im = im.convert("RGB")
            im.thumbnail((512, 512))
            return str(imagehash.phash(im))
    except Exception:
        return None


unknown = set(HOLDOUT_SOURCES) - {s.name for s in SOURCES}
assert not unknown, f"HOLDOUT_SOURCES names not in SOURCES: {unknown}"

# 1. Sample and balance, without the held-out sources
skipped = Counter()
selected = {}
for category, cap in (("face", FACE_CAP), ("general", GENERAL_CAP)):
    for label in ("REAL", "AI_GENERATED"):
        candidates = [(s, p) for s, p in pools[(category, label)] if s not in HOLDOUT_SOURCES]
        selected[(category, label)] = round_robin(candidates, cap, skipped)
        print(f"  {category} {label}: picked {len(selected[(category, label)]):,}")
    n = min(len(selected[(category, "REAL")]), len(selected[(category, "AI_GENERATED")]))
    for label in ("REAL", "AI_GENERATED"):
        selected[(category, label)] = selected[(category, label)][:n]
    print(f"{category:8s}{n:,} images per class")
# 2. Split; held-out sources go to test only
assigned = []   # (split, label, category, source, path, holdout)
for (category, label), items in sorted(selected.items()):
    rng.shuffle(items)
    a, b = int(len(items) * SPLIT[0]), int(len(items) * (SPLIT[0] + SPLIT[1]))
    for split, part in (("train", items[:a]), ("val", items[a:b]), ("test", items[b:])):
        assigned += [(split, label, category, source, path, False) for source, path in part]
for (category, label), items in sorted(pools.items()):
    held = [(s, p) for s, p in items if s in HOLDOUT_SOURCES]
    assigned += [("test", label, category, s, p, True) for s, p in round_robin(held, HOLDOUT_CAP, skipped)]
if skipped:
    print("Skipped as too small, too large or unreadable:", dict(skipped))

# 3. Remove val/test images that duplicate an image in an earlier split
print(f"Hashing {len(assigned):,} images...")
with ThreadPoolExecutor(8) as pool:
    hashes = list(tqdm(pool.map(phash, (row[4] for row in assigned)), total=len(assigned)))

earlier = {h for row, h in zip(assigned, hashes) if row[0] == "train" and h}
kept, duplicates, unreadable = [], Counter(), Counter()
for split in ("train", "val", "test"):
    for row, h in zip(assigned, hashes):
        if row[0] != split:
            continue
        if h is None:                      # couldn't be fully decoded (e.g. truncated): training would crash on it
            unreadable[row[3]] += 1
            continue
        if split != "train":
            if h in earlier:
                duplicates[split] += 1
                continue
            earlier.add(h)
        kept.append(row)
print("Removed as duplicates of an earlier image:", dict(duplicates) or "none")
print("Removed as unreadable when decoded:", dict(unreadable) or "none")

# 4. Write the folders train.py reads: one copy per image, at most 512 px on its longest side.
#    Training resizes to 224 anyway, and small files make every epoch many times faster.
SHRINK_TO = 512


def write_copy(job):
    src, dst = job
    try:
        with Image.open(src) as im:
            im.draft("RGB", (SHRINK_TO, SHRINK_TO))   # JPEGs decode straight at reduced size
            im = im.convert("RGB")
            im.thumbnail((SHRINK_TO, SHRINK_TO))
            im.save(dst, "JPEG", quality=95)          # training re-compresses every image anyway
        return True
    except Exception:
        return False


if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
rows, jobs = [], []
for split, label, category, source, path, holdout in kept:
    out_dir = DATA_DIR / split / label
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{category}_{source}_{len(rows):06d}.jpg"
    jobs.append((path, out))
    rows.append({"split": split, "label": label, "category": category, "source": source,
                 "holdout": holdout, "link": str(out), "original": str(path)})

print(f"Writing {len(jobs):,} resized copies...")
with ThreadPoolExecutor(8) as pool:
    written = list(tqdm(pool.map(write_copy, jobs), total=len(jobs)))
rows = [row for row, ok in zip(rows, written) if ok]
print(f"Could not write: {written.count(False)}")

with open(DATA_DIR / "manifest.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

for split in ("train", "val", "test"):
    classes = sorted(d.name for d in (DATA_DIR / split).iterdir() if d.is_dir())
    assert classes == ["AI_GENERATED", "REAL"], f"{split}: unexpected classes {classes}"
manifest = pd.read_csv(DATA_DIR / "manifest.csv")
print(manifest.groupby(["split", "holdout", "category", "label"]).size().unstack("label").to_string())

    

## 3. Train

Runs the repository's `scripts/train.py` unchanged. The cell fails if training fails, so *Run All* never continues with a missing or stale model.

In [ ]:
import sys

env = {**os.environ, "PYTHONPATH": ".", "MLFLOW_TRACKING_URI": "file:./mlruns",
       "MLFLOW_ALLOW_FILE_STORE": "true", "TQDM_DISABLE": "1"}
proc = subprocess.Popen([sys.executable, "scripts/train.py"], env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, f"train.py exited with code {proc.returncode}"

## 4. Check the export

The ONNX file is what the API serves, so it must give the same answers as the PyTorch model it came from. This cell compares both on 32 test images and checks that the ONNX model accepts a batch.

In [ ]:
import json

import numpy as np
import onnxruntime as ort
import torch

from scripts.train import build_model, get_transforms

meta = json.loads(Path("models/metadata.json").read_text())
assert meta["classes"] == ["AI_GENERATED", "REAL"], f"Unexpected classes: {meta['classes']}"

session = ort.InferenceSession("models/model_v1.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
dynamic = not isinstance(session.get_inputs()[0].shape[0], int)
print("ONNX input shape:", session.get_inputs()[0].shape,
      "(dynamic batch)" if dynamic else "(FIXED batch of 1: apply the dynamic_axes fix in train.py)")

model = build_model(2, freeze_backbone=False)
checkpoint = torch.load("models/best_checkpoint.pt", map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
_, val_tf = get_transforms()


def onnx_probs(batch):
    """Softmax probabilities from the ONNX model; one image at a time if the batch is fixed."""
    if dynamic:
        logits = session.run(None, {input_name: batch})[0]
    else:
        logits = np.concatenate([session.run(None, {input_name: batch[i:i + 1]})[0] for i in range(len(batch))])
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


sample = manifest[manifest.split == "test"].sample(32, random_state=SEED)
x = torch.stack([val_tf(Image.open(p).convert("RGB")) for p in sample.link])
with torch.no_grad():
    torch_probs = torch.softmax(model(x), dim=1).numpy()
diff = float(np.abs(onnx_probs(x.numpy()) - torch_probs).max())
print(f"Largest probability difference, ONNX vs PyTorch: {diff:.1e}")
assert diff < 1e-3, "The ONNX export does not match the PyTorch model"
if dynamic:
    assert session.run(None, {input_name: x[:4].numpy()})[0].shape == (4, 2)
    print("A batch of 4 runs.")

## 5. Evaluate on the test set

These images were never used for training or for choosing the best epoch, so these are the numbers for the model card. Images from `HOLDOUT_SOURCES` are reported separately as the *unseen generator* result. Two views:

* **Accuracy** treats the model as a plain classifier (P(AI) of at least 0.5 means AI).
* **The product view** uses the API's thresholds. Verdicts between them are "uncertain", so it reports how often the model decides (*decided share*) and how accurate it is when it does.

In [ ]:
from sklearn.metrics import f1_score

from src.config import settings

upper = settings.MODEL_THRESHOLD
lower = settings.MODEL_THRESHOLD - settings.GRAY_AREA_MARGIN


def p_ai_for(df, damage=None):
    """P(AI) from the ONNX model for every image in df, optionally after damaging each image."""
    out = []
    for i in range(0, len(df), 64):
        images = [Image.open(p).convert("RGB") for p in df.link[i:i + 64]]
        if damage:
            images = [damage(im) for im in images]
        out.append(onnx_probs(np.stack([val_tf(im).numpy() for im in images]))[:, 0])   # class 0 is AI_GENERATED
    return np.concatenate(out) if out else np.array([])


def with_predictions(df, damage=None):
    df = df.reset_index(drop=True).copy()
    df["p_ai"] = p_ai_for(df, damage)
    df["is_ai"] = df.label == "AI_GENERATED"
    df["pred_ai"] = df.p_ai >= 0.5
    df["correct"] = df.pred_ai == df.is_ai
    return df


def summarize(df, upper, lower):
    """Plain accuracy, plus the product view: how often it decides, and how often it is right then."""
    says_ai, says_real = df.p_ai >= upper, df.p_ai <= lower
    decided = says_ai | says_real
    right = (says_ai & df.is_ai) | (says_real & ~df.is_ai)
    return {"images": len(df), "accuracy": round(float(df.correct.mean()), 4),
            "decided_share": round(float(decided.mean()), 4),
            "accuracy_when_decided": round(float(right[decided].mean()), 4) if decided.any() else None}


test_all = with_predictions(manifest[manifest.split == "test"])
test = test_all[~test_all.holdout].reset_index(drop=True)      # generators seen in training
unseen = test_all[test_all.holdout].reset_index(drop=True)     # HOLDOUT_SOURCES only

overall = {**summarize(test, upper, lower),
           "f1_weighted": round(float(f1_score(test.is_ai, test.pred_ai, average="weighted")), 4)}
by_category = {c: summarize(g, upper, lower) for c, g in test.groupby("category")}
by_source = pd.DataFrame({k: summarize(g, upper, lower)
                          for k, g in test_all.groupby(["holdout", "category", "source", "label"])}).T
unseen_summary = summarize(unseen, upper, lower) if len(unseen) else None

print("Overall (seen generators):", overall)
print("Unseen generator:", unseen_summary or "no HOLDOUT_SOURCES set")
print(pd.DataFrame(by_category).T.to_string())
print(by_source.to_string())

Path("reports").mkdir(exist_ok=True)
by_source.to_csv("reports/test_by_source.csv")
meta["evaluation"] = {"test_set": overall, "unseen_generator": unseen_summary,
                      "by_category": by_category, "thresholds": {"ai": upper, "real": lower}}
meta["provenance"] = {
    "git_commit": GIT_COMMIT, "seed": SEED, "split": list(SPLIT), "min_side": MIN_SIDE,
    "holdout_sources": list(HOLDOUT_SOURCES), "duplicates_removed": dict(duplicates),
    "torch": torch.__version__, "onnxruntime": ort.__version__,
    "images": {f"{s}/{l}": int(n) for (s, l), n in manifest.groupby(["source", "label"]).size().items()},
}

## 6. Choose the uncertainty thresholds

The printed values are a suggestion from validation data only. Compare them with the held-out rows before changing anything on Render. For v2.1.0 they were not used; see "Choosing the thresholds" in docs/MODEL_CARD.md.

In [ ]:
import math

MIN_UNCERTAIN_WIDTH = 0.25   # the uncertain zone never gets narrower than this


def lowest_threshold(scores, positive, target, min_n=50):
    """Lowest t such that "score >= t" is right at least `target` of the time, over at least min_n images."""
    order = np.argsort(-scores)
    s, pos = scores[order], positive[order]
    n = np.arange(1, len(s) + 1)
    ok = (np.cumsum(pos) / n >= target) & (n >= min_n)
    return float(s[np.where(ok)[0].max()]) if ok.any() else None


val = with_predictions(manifest[manifest.split == "val"])
p, is_ai = val.p_ai.to_numpy(), val.is_ai.to_numpy()
t_ai = lowest_threshold(p, is_ai, TARGET_PRECISION)
t_real = lowest_threshold(1 - p, ~is_ai, TARGET_PRECISION)

# Round outwards, so the rounded zones are never less precise than the measured ones
new_upper = math.ceil(t_ai * 100) / 100 if t_ai is not None else upper
new_lower = math.floor((1 - t_real) * 100) / 100 if t_real is not None else lower
if t_ai is None or t_real is None:
    print(f"Could not reach {TARGET_PRECISION:.0%} precision on at least 50 validation images for one side; "
          "kept the current value there. Lower TARGET_PRECISION or add data.")

# The validation set only holds familiar kinds of images, where the model is rarely unsure, so on its own
# it would happily shrink the uncertain zone to nothing. Unfamiliar images need that margin: keep a minimum.
if new_upper - new_lower < MIN_UNCERTAIN_WIDTH:
    middle = (new_upper + new_lower) / 2
    new_upper = min(0.99, round(middle + MIN_UNCERTAIN_WIDTH / 2, 2))
    new_lower = max(0.01, round(middle - MIN_UNCERTAIN_WIDTH / 2, 2))
    print(f"Widened the uncertain zone to {MIN_UNCERTAIN_WIDTH:.2f}: "
          "thresholds tuned on familiar images can't see how the model does on unfamiliar ones.")

# Compare on familiar test images AND on the held-out sources, which are never used to choose anything
comparison = {
    f"current (AI >= {upper:.2f}, real <= {lower:.2f})": summarize(test, upper, lower),
    f"suggested (AI >= {new_upper:.2f}, real <= {new_lower:.2f})": summarize(test, new_upper, new_lower),
}
if len(unseen):
    comparison["current, held-out sources"] = summarize(unseen, upper, lower)
    comparison["suggested, held-out sources"] = summarize(unseen, new_upper, new_lower)
print(pd.DataFrame(comparison).T.to_string())
print(f"\nSuggested for Render:  MODEL_THRESHOLD={new_upper:.2f}  GRAY_AREA_MARGIN={new_upper - new_lower:.2f}")
print("A suggestion, not a decision: if it decides more often but is right less often, keep the current values.")

meta["recommended_thresholds"] = {
    "MODEL_THRESHOLD": new_upper, "GRAY_AREA_MARGIN": round(new_upper - new_lower, 2),
    "target_precision": TARGET_PRECISION, "min_uncertain_width": MIN_UNCERTAIN_WIDTH,
    "chosen_on": "validation set", "also_reported_on": "held-out sources",
}
meta["evaluation"]["test_set_with_recommended_thresholds"] = summarize(test, new_upper, new_lower)
if len(unseen):
    meta["evaluation"]["held_out_with_recommended_thresholds"] = summarize(unseen, new_upper, new_lower)

## 7. Robustness: the same test images after real-world damage

Images on the web are re-compressed, resized and sometimes softened. This scores a sample of the test set again after each kind of damage, with the chosen thresholds. Training is unchanged; this is evaluation only. A large drop under blur means the model relies on fine detail, which is why blur is not used as training augmentation.

In [ ]:
import io

from PIL import ImageFilter


def jpeg_q75(im):
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=75)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def half_size(im):
    w, h = im.size
    return im.resize((max(1, w // 2), max(1, h // 2)), Image.BILINEAR).resize((w, h), Image.BILINEAR)


def blur_r1(im):
    return im.filter(ImageFilter.GaussianBlur(1))


subset = test.sample(min(ROBUSTNESS_SAMPLE, len(test)), random_state=SEED)
robustness = pd.DataFrame([
    {"condition": name, **summarize(with_predictions(subset, damage), new_upper, new_lower)}
    for name, damage in (("clean", None), ("jpeg_q75", jpeg_q75), ("half_size", half_size), ("blur_r1", blur_r1))
])
print(robustness.to_string(index=False))
robustness.to_csv("reports/robustness.csv", index=False)
meta["evaluation"]["robustness"] = robustness.to_dict(orient="records")

## 8. What is the model looking at?

GradCAM heatmaps for test images of each class: four the model got right and up to four it got wrong. Look for shortcuts: heat on watermarks, logos, text, borders or corners instead of the subject means the data has a pattern the model is cheating with. The grid is saved for the model card.

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

panels = []
for label in ("AI_GENERATED", "REAL"):
    g = test[test.label == label]
    for part in (g[g.correct], g[~g.correct]):
        panels += list(part.sample(min(4, len(part)), random_state=SEED).itertuples())

fig, axes = plt.subplots(4, 4, figsize=(12, 13))
with GradCAM(model=model, target_layers=[model.features[-1]]) as cam:
    for ax, row in zip(axes.flat, panels):
        img = Image.open(row.link).convert("RGB")
        chosen = 0 if row.pred_ai else 1   # explain the class the model picked (0 = AI_GENERATED)
        heat = cam(input_tensor=val_tf(img).unsqueeze(0), targets=[ClassifierOutputTarget(chosen)])[0]
        rgb = np.asarray(img.resize((224, 224)), dtype=np.float32) / 255
        ax.imshow(show_cam_on_image(rgb, heat, use_rgb=True))
        mark = "right" if row.correct else "WRONG"
        said = "AI" if row.pred_ai else "real"
        ax.set_title(f"{mark}: {row.label} -> said {said} ({row.p_ai:.2f})\n{row.source}", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig("reports/gradcam_grid.png", dpi=110)
plt.show()

## 9. Package the model

Writes SHA-256 checksums into `metadata.json`, so the API can refuse a file that was swapped or corrupted, then zips everything to download. `model_v1.onnx.data` holds the weights and must always travel with `model_v1.onnx`.

In [ ]:
import hashlib
import zipfile


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


model_files = [f for f in ("model_v1.onnx", "model_v1.onnx.data", "best_checkpoint.pt") if (Path("models") / f).exists()]
meta["sha256"] = {f: sha256(Path("models") / f) for f in model_files}
Path("models/metadata.json").write_text(json.dumps(meta, indent=4))

out = Path("/kaggle/working/aon_model.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [*model_files, "metadata.json"]:
        z.write(Path("models") / f, f"models/{f}")
    for report in sorted(Path("reports").glob("*")):
        z.write(report, f"reports/{report.name}")
    z.write(DATA_DIR / "manifest.csv", "reports/manifest.csv")
    
print(f"{out} ({out.stat().st_size / 1e6:.1f} MB): download it from the Output tab and copy models/ into the repo.")
print(json.dumps({k: meta["evaluation"][k] for k in ("test_set", "unseen_generator", "test_set_with_recommended_thresholds")}, indent=2))
print(json.dumps(meta["recommended_thresholds"], indent=2))

## Before you commit

* Compare the test F1 with the numbers in the README and model card, and update them from `metadata.json`.
* Put into `MODEL_CARD.md`: the per-category table, the weakest sources (`reports/test_by_source.csv`), the unseen-generator result, the decided share and accuracy when decided, the robustness table, the duplicates removed and the GradCAM grid.
* Set `MODEL_THRESHOLD` and `GRAY_AREA_MARGIN` on Render (and in `.env.example`) from `recommended_thresholds`.
* Commit `model_v1.onnx`, `model_v1.onnx.data`, `best_checkpoint.pt` and `metadata.json` together.